# Gold Layer — E-commerce


In [ ]:
CREATE SCHEMA IF NOT EXISTS azure_databricks_dev.gold;


In [ ]:
-- Customer dimension
CREATE OR REPLACE TABLE azure_databricks_dev.gold.dim_users
USING DELTA
AS
SELECT
  user_id,
  name,
  email,
  gender,
  city,
  signup_date
FROM azure_databricks_dev.silver.users;


In [ ]:
-- Product dimension
CREATE OR REPLACE TABLE azure_databricks_dev.gold.dim_products
USING DELTA
AS
SELECT
  product_id,
  product_name,
  category,
  brand,
  price,
  rating
FROM azure_databricks_dev.silver.products;


In [ ]:
-- Order fact
CREATE OR REPLACE TABLE azure_databricks_dev.gold.fact_orders
USING DELTA
AS
SELECT
  o.order_id,
  o.user_id,
  CAST(o.order_date AS DATE) AS order_date,
  YEAR(o.order_date) AS order_year,
  MONTH(o.order_date) AS order_month,
  o.order_status,
  o.total_amount
FROM azure_databricks_dev.silver.orders o
INNER JOIN azure_databricks_dev.silver.users u
  ON o.user_id = u.user_id;


In [ ]:
-- Order-item fact
CREATE OR REPLACE TABLE azure_databricks_dev.gold.fact_order_items
USING DELTA
AS
SELECT
  oi.order_item_id,
  oi.order_id,
  oi.user_id,
  oi.product_id,
  oi.quantity,
  oi.item_price,
  oi.item_total,
  o.order_date,
  o.order_status
FROM azure_databricks_dev.silver.order_items oi
INNER JOIN azure_databricks_dev.silver.orders o
  ON oi.order_id = o.order_id
INNER JOIN azure_databricks_dev.silver.products p
  ON oi.product_id = p.product_id;


In [ ]:
-- Daily sales summary for dashboards/BI
CREATE OR REPLACE TABLE azure_databricks_dev.gold.daily_sales
USING DELTA
AS
SELECT
  CAST(order_date AS DATE) AS sales_date,
  COUNT(DISTINCT order_id) AS order_count,
  COUNT(DISTINCT user_id) AS customer_count,
  SUM(total_amount) AS revenue,
  AVG(total_amount) AS average_order_value
FROM azure_databricks_dev.silver.orders
WHERE order_status IN ('completed', 'processing')
GROUP BY CAST(order_date AS DATE);


In [ ]:
-- Product performance summary
CREATE OR REPLACE TABLE azure_databricks_dev.gold.product_performance
USING DELTA
AS
SELECT
  p.product_id,
  p.product_name,
  p.category,
  p.brand,
  COUNT(DISTINCT oi.order_id) AS order_count,
  SUM(oi.quantity) AS units_sold,
  SUM(oi.item_total) AS revenue,
  AVG(r.rating) AS average_review_rating
FROM azure_databricks_dev.silver.products p
LEFT JOIN azure_databricks_dev.silver.order_items oi
  ON p.product_id = oi.product_id
LEFT JOIN azure_databricks_dev.silver.reviews r
  ON p.product_id = r.product_id
GROUP BY
  p.product_id,
  p.product_name,
  p.category,
  p.brand;


In [ ]:
-- Customer sales summary
CREATE OR REPLACE TABLE azure_databricks_dev.gold.customer_sales
USING DELTA
AS
SELECT
  u.user_id,
  u.name,
  u.city,
  COUNT(DISTINCT o.order_id) AS order_count,
  SUM(o.total_amount) AS total_spend,
  AVG(o.total_amount) AS average_order_value,
  MAX(o.order_date) AS last_order_date
FROM azure_databricks_dev.silver.users u
LEFT JOIN azure_databricks_dev.silver.orders o
  ON u.user_id = o.user_id
GROUP BY
  u.user_id,
  u.name,
  u.city;


In [ ]:
-- Gold validation / preview
SELECT * FROM azure_databricks_dev.gold.daily_sales
ORDER BY sales_date DESC
LIMIT 20;
